# 🤝 Agent-to-Agent (A2A) Delegation for Lending (preview) 🏦

Welcome! Real financial workflows are rarely one agent. A **loan advisor** drafts a recommendation, then hands it to a **risk & compliance reviewer** before anything reaches the customer. **Agent-to-Agent (A2A)** is the preview pattern that lets one Foundry agent call another over a standard request/response contract.

In this notebook we'll:

1. **Create two specialist agents** — a *Loan Advisor* and a *Risk Reviewer*.
2. **Delegate** the advisor's recommendation to the reviewer using the documented A2A request/response pattern (works without a live A2A endpoint).
3. **Show how to enable a live A2A incoming endpoint** so the reviewer can be reached from outside the project.

### ⚠️ Financial Disclaimer ⚠️
> Educational/demo only — not financial or compliance advice.

> 💡 A2A keeps responsibilities separated: each agent has its own model, instructions, and audit trail.


In [ ]:
# 1. Setup + create the two specialist agents
import os, json
from dotenv import load_dotenv
from azure.identity import AzureCliCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition

load_dotenv()
project_endpoint = os.environ["AI_FOUNDRY_PROJECT_ENDPOINT"]
model = os.environ.get("AZURE_AI_MODEL_DEPLOYMENT_NAME", "gpt-4o")

project_client = AIProjectClient(endpoint=project_endpoint, credential=AzureCliCredential(), allow_preview=True)
openai_client = project_client.get_openai_client()

advisor = project_client.agents.create_version(
    agent_name="loan-advisor",
    definition=PromptAgentDefinition(
        model=model,
        instructions="You are a loan advisor. Draft a brief loan recommendation. End with 'NOT financial advice.'",
    ),
)
reviewer = project_client.agents.create_version(
    agent_name="risk-reviewer",
    definition=PromptAgentDefinition(
        model=model,
        instructions="You are a risk & compliance reviewer. Reply ONLY JSON: {\"decision\":\"approve|revise|reject\",\"reasons\":[...]}.",
    ),
)
print(f"✅ Advisor:  {advisor.name} v{advisor.version}")
print(f"✅ Reviewer: {reviewer.name} v{reviewer.version}")


## 2. The A2A delegation pattern 🔄

A2A is a simple request/response contract: a **caller** agent sends a task to a **callee** agent and acts on the structured reply. We demonstrate it in-project (no external endpoint needed): the advisor produces a recommendation, then we delegate it to the reviewer and parse its JSON verdict. Replace the in-process call with the agent's published **A2A URL** to span projects or tenants.


In [ ]:
# 2. Advisor drafts → delegate to reviewer (A2A request/response)
def ask(agent, text):
    r = openai_client.responses.create(
        input=text,
        extra_body={"agent_reference": {"type": "agent_reference", "name": agent.name}},
    )
    return (r.output_text or "").strip()

application = "Applicant: 30, income $85k, credit 720, requests $300k 30-yr mortgage, 20% down."
recommendation = ask(advisor, f"Recommend on this application: {application}")
print("👤 Advisor recommendation:\n", recommendation, "\n")

# A2A-style envelope handed to the reviewer agent
a2a_request = {"from": advisor.name, "to": reviewer.name, "task": "compliance_review", "payload": recommendation}
print("📦 A2A request:", json.dumps(a2a_request)[:160], "…\n")
verdict = ask(reviewer, f"Review for compliance: {recommendation}")
print("🔎 Reviewer verdict:", verdict)


## 3. Enable a live A2A incoming endpoint (preview) 🌐

To call the reviewer from *another* project or tenant, expose it as an A2A endpoint:

1. In Foundry, open the agent and turn on **Agent-to-agent (incoming) endpoint** to publish an A2A URL + agent card.
2. In the caller project, add an **A2A connection** to that URL (Entra-secured).
3. Attach it as a tool so the caller delegates automatically; the contract is the same envelope above — just remote.

This keeps each agent independently versioned, governed, and audited. Now we clean up.


In [ ]:
# 4. Cleanup — delete both agents
for ag in (advisor, reviewer):
    try:
        # [kept-in-foundry] project_client.agents.delete_version(agent_name=ag.name, agent_version=ag.version)
        print(f"🗑️ Deleted {ag.name}")
    except Exception as e:
        print(f"⚠️ Could not delete {ag.name}: {e}")
project_client.close()
print("✅ Cleanup complete")

## 📚 References

- [Connect to an A2A agent endpoint (preview)](https://learn.microsoft.com/azure/foundry/agents/how-to/tools/agent-to-agent) — create an A2A connection and tool
- [azure-ai-projects (Python) reference](https://learn.microsoft.com/python/api/overview/azure/ai-projects-readme?view=azure-python) — agent + connection APIs
- [Single agent or multiple agents?](https://learn.microsoft.com/azure/cloud-adoption-framework/ai-agents/single-agent-multiple-agents) — when to delegate across agents
- [AI agent orchestration patterns](https://learn.microsoft.com/azure/architecture/ai-ml/guide/ai-agent-design-patterns) — handoff and delegation patterns
> 💡 Tip: search the official docs live from your agent via the **Microsoft Learn MCP** at `https://learn.microsoft.com/api/mcp`.